In [ ]:
import itertools
import pulp as plp

In [ ]:
# Constants

ROWS = COLUMNS = VALUES = range(1, 10)
SQUARE_INDICES = list(itertools.product(ROWS, COLUMNS, VALUES))
BOXES = [
    # indices for single box
    [(3 * l + i + 1, 3 * k + j + 1) for i in range(3) for j in range(3)]
    # indices for boxes along columns
    for k in range(3)
    # indices for boxes along rows
    for l in range(3)
]

INITIAL_CONDITIONS = [
    (1, 1, 5),
    (2, 1, 6),
    (4, 1, 8),
    (5, 1, 4),
    (6, 1, 7),
    (1, 2, 3),
    (3, 2, 9),
    (7, 2, 6),
    (3, 3, 8),
    (2, 4, 1),
    (5, 4, 8),
    (8, 4, 4),
    (1, 5, 7),
    (2, 5, 9),
    (4, 5, 6),
    (6, 5, 2),
    (8, 5, 1),
    (9, 5, 8),
    (2, 6, 5),
    (5, 6, 3),
    (8, 6, 9),
    (7, 7, 2),
    (3, 8, 6),
    (7, 8, 8),
    (9, 8, 7),
    (4, 9, 3),
    (5, 9, 1),
    (6, 9, 6),
    (8, 9, 5),
]

In [ ]:
# Problem
sudoku = plp.LpProblem(name="sudoku")

# Decision variables
squares = plp.LpVariable.dicts("squares", indices=SQUARE_INDICES, cat="Binary")

# Constraints
for r in ROWS:
    for c in COLUMNS:
        sudoku += (
            plp.lpSum([squares[(r, c, k)] for k in VALUES]) == 1,
            f"only_one_number_for_square_{r, c}",
        )

for v in VALUES:
    for r in ROWS:
        sudoku += (
            plp.lpSum([squares[(r, c, v)] for c in COLUMNS]) == 1,
            f"only_one_square_with_value_{v}_in_row_{r}",
        )

    for c in COLUMNS:
        sudoku += (
            plp.lpSum([squares[(r, c, v)] for r in ROWS]) == 1,
            f"only_one_square_with_value_{v}_in_column_{c}",
        )

    for box in BOXES:
        sudoku += (
            plp.lpSum([squares[(r, c, v)] for (r, c) in box]) == 1,
            f"only_one_square_with_value_{v}_in_box_{BOXES.index(box) + 1}",
        )

for i in INITIAL_CONDITIONS:
    sudoku += (squares[i] == 1, f"keep_initial_value_for_square_{i}")

In [ ]:
# The problem data is written to an .lp file
sudoku.writeLP("sudoku.lp")

# The problem is solved using PuLP's choice of Solver
sudoku.solve()

# The status of the solution is printed to the screen
print("Status:", plp.LpStatus[sudoku.status])

In [ ]:
# Write solution to file
sudokuout = open("sudokuout.txt", "w")

for r in ROWS:
    if r in [1, 4, 7]:
        sudokuout.write("+-------+-------+-------+\n")
    for c in COLUMNS:
        for v in VALUES:
            if plp.value(squares[(r, c, v)]) == 1:
                if c in [1, 4, 7]:
                    sudokuout.write("| ")
                sudokuout.write(str(v) + " ")
                if c == 9:
                    sudokuout.write("|\n")
sudokuout.write("+-------+-------+-------+")
sudokuout.close()